## Setup

First, we need to install the required packages and set up SQL magic for Jupyter notebooks.

In [ ]:
# Idempotent notebook-dep installer (safe to rerun).
# Uses uv from the Codespace; falls back to pip if running standalone.
import importlib.util, subprocess, sys, shutil
_pkgs = {'sql': 'jupysql', 'duckdb': 'duckdb',
         'duckdb_engine': 'duckdb-engine', 'prettytable': 'prettytable'}
_missing = [p for m, p in _pkgs.items() if importlib.util.find_spec(m) is None]
if _missing:
    cmd = (['uv', 'pip', 'install', '--python', sys.executable]
           if shutil.which('uv')
           else [sys.executable, '-m', 'pip', 'install'])
    subprocess.check_call(cmd + _missing)
    print('Installed:', _missing)
else:
    print('All notebook deps already present.')


In [2]:
# Load SQL magic extension
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

## Connection Options

DuckDB supports different connection modes:

### 1. In-Memory Database (Temporary)
Data exists only while the connection is open. Perfect for exploration.

In [3]:
# Connect to in-memory database
%sql duckdb:///:memory:

### 2. Persistent Database (File-based)
Data is saved to disk. Uncomment to use:

```python
%sql duckdb:///my_database.db
```

## Your First Query

Let's test the connection with a simple query.

In [4]:
%%sql
SELECT 'Hello DuckDB!' AS greeting, 42 AS answer, CURRENT_TIMESTAMP AS current_time;

,greeting,answer,current_time
0,Hello DuckDB!,42,2025-11-27 07:05:41.473111+01:00


## SQL Magic Syntax

There are two ways to run SQL in Jupyter notebooks:

### Single-line: `%sql`
Use for simple, one-line queries:

In [5]:
%sql SELECT 1 + 1 AS result;

,result
0,2


### Multi-line: `%%sql`
Use for complex queries spanning multiple lines:

In [6]:
%%sql
SELECT 
    'Multi-line' AS type,
    'query' AS description,
    'example' AS purpose;

,type,description,purpose
0,Multi-line,query,example


## Create Your First Table

Let's create a simple table to demonstrate DuckDB's capabilities.

In [7]:
%%sql
CREATE TABLE demo (
    id INTEGER,
    name VARCHAR,
    value DOUBLE
);

INSERT INTO demo VALUES 
    (1, 'Alpha', 10.5),
    (2, 'Beta', 20.3),
    (3, 'Gamma', 15.8);

SELECT * FROM demo;

,id,name,value
0,1,Alpha,10.5
1,2,Beta,20.3
2,3,Gamma,15.8


## DuckDB System Information

Let's check the DuckDB version and settings:

In [8]:
%%sql
SELECT version() AS duckdb_version;

,duckdb_version
0,v1.4.2


In [9]:
%%sql
-- List all tables in the database
SHOW TABLES;

,Success


## Quick Performance Demo

DuckDB is fast! Let's generate and query 100,000 rows:

In [10]:
%%sql
-- Generate 100,000 rows using DuckDB's range function
SELECT 
    COUNT(*) AS total_rows,
    SUM(value) AS sum_values,
    AVG(value) AS avg_value
FROM (
    SELECT 
        range AS id,
        'Item_' || range AS name,
        range * 1.5 AS value
    FROM range(100000)
);

,total_rows,sum_values,avg_value
0,100000,7.499925e+09,74999.25


## DuckDB vs Other Databases

| Feature | DuckDB | SQLite | PostgreSQL | Pandas |
|---------|--------|--------|------------|--------|
| **Purpose** | Analytics (OLAP) | Transactions (OLTP) | General purpose | Data analysis |
| **Speed (Analytics)** | 🚀🚀🚀 Very Fast | 🐌 Slow | 🚀 Fast | 🚀🚀 Fast |
| **Setup** | None | None | Server required | None |
| **File size** | Small | Small | Large | Medium |
| **SQL Standard** | PostgreSQL | SQLite dialect | PostgreSQL | Limited |
| **Direct file query** | ✅ CSV, Parquet, JSON | ❌ | ❌ | ✅ CSV, Excel |
| **Larger than RAM** | ✅ Yes | ❌ Limited | ✅ Yes | ❌ No |

## Use Cases

DuckDB excels at:

1. **Data Exploration**: Quick ad-hoc analysis without loading data
2. **ETL Pipelines**: Transform data efficiently
3. **Embedded Analytics**: Analytics in applications without a separate database server
4. **Data Science**: SQL-based data manipulation in notebooks
5. **Large File Analysis**: Query CSV/Parquet files larger than RAM
6. **Prototyping**: Test queries before deploying to production databases

## Key Features Preview

Here's a taste of what DuckDB can do (we'll explore these in detail in later notebooks):

In [11]:
%%sql
-- Read CSV directly without importing
SELECT * FROM '../sample_data/states.csv' LIMIT 3;

,id,name,country_id,country_code,country_name,state_code,type,latitude,longitude
0,3901,Badakhshan,1,AF,Afghanistan,BDS,None,36.734772,70.811995
1,3871,Badghis,1,AF,Afghanistan,BDG,None,35.167134,63.769538
2,3875,Baghlan,1,AF,Afghanistan,BGL,None,36.178903,68.745306


In [12]:
%%sql
-- Complex analytical query
SELECT 
    id,
    name,
    value,
    AVG(value) OVER () AS avg_all,
    value - AVG(value) OVER () AS diff_from_avg
FROM demo;

,id,name,value,avg_all,diff_from_avg
0,1,Alpha,10.5,15.533333,-5.033333
1,2,Beta,20.3,15.533333,4.766667
2,3,Gamma,15.8,15.533333,0.266667


## Clean Up

Drop the demo table:

In [13]:
%%sql
DROP TABLE IF EXISTS demo;

,Success
